# Z3-19 -- Optimisation multi-objectif (lexicographique, Pareto, somme pondérée)

**Navigation** : [Index Z3-Python](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [Série Z3 C#](../Z3-Linq2Z3/README.md) | [<< 18 - Sudoku Modes](Z3-18-Sudoku-Modes-Python.ipynb) | [20 - Machine de Gödel >>](Z3-20-Machine-de-Goedel-Python.ipynb)

Quand un problème d'optimisation a **plusieurs critères contradictoires** (coût vs impact, latence vs débit, performance vs énergie), il n'y a souvent **pas une seule solution optimale** mais un **ensemble de compromis**. Z3 expose `Optimize()`, le solveur dédié à l'optimisation sous contraintes, qui supporte trois familles de stratégies complémentaires :

- **Lexicographique** : on optimise les critères dans un ordre donné (impact d'abord, puis coût, etc.). Une seule   solution sort -- celle qui est meilleure sur le premier critère, et parmi celles-ci la meilleure sur le deuxième, etc.
- **Pareto (front de Pareto)** : on énumère toutes les solutions **non dominées** -- aucune autre solution n'est   strictement meilleure sur tous les critères simultanément. Le résultat est un *ensemble* (souvent petit) qui   représente l'ensemble des compromis rationnels.
- **Somme pondérée** : on agrège les critères en une seule fonction `somme(poids_i * critere_i)`, qu'on maximise   (ou minimise). Le poids de chaque critère reflète l'arbitrage du décideur. Cette méthode est simple mais   **manque certains points du front de Pareto** si la fonction objectif n'est pas concave.

## Problème jouet

On optimise la sélection de **3 projets** sous un **budget de 12 k€** :

| Projet | Coût (k€) | Impact (utilité) |
|--------|----------|-----------------|
| A      | 4        | 6               |
| B      | 5        | 8               |
| C      | 7        | 10              |

Objectifs contradictoires : maximiser l'impact total et minimiser le coût total. Le solveur `Optimize()` de Z3 permet d'encoder ces trois stratégies en quelques lignes et de comparer les solutions obtenues.

In [1]:
from z3 import Optimize, Int, Sum, If, Or, sat, set_param
set_param('verbose', 0)  # silence le solveur pour la lisibilite des sorties

# --- Definition du probleme jouet ---
couts = {'A': 4, 'B': 5, 'C': 7}
impacts = {'A': 6, 'B': 8, 'C': 10}
BUDGET = 12

# Variables de decision : x[p] = 1 si on selectionne le projet p, 0 sinon
x = {p: Int(f'x_{p}') for p in 'ABC'}
opt = Optimize()
for p in 'ABC':
    opt.add(0 <= x[p], x[p] <= 1)
opt.add(Sum([x[p] * couts[p] for p in 'ABC']) <= BUDGET)

cout_total = Sum([x[p] * couts[p] for p in 'ABC'])
impact_total = Sum([x[p] * impacts[p] for p in 'ABC'])

print('Contraintes posees : chaque projet x_p in {0,1}, somme des couts <= 12 k€')
print('Objectifs : maximiser impact_total, minimiser cout_total')

Contraintes posees : chaque projet x_p in {0,1}, somme des couts <= 12 k€
Objectifs : maximiser impact_total, minimiser cout_total


In [2]:
# Mesure 1 -- optimisation lexicographique : maximiser l'impact d'abord, puis minimiser le cout
opt_lex = Optimize()
for p in 'ABC':
    opt_lex.add(0 <= x[p], x[p] <= 1)
opt_lex.add(Sum([x[p] * couts[p] for p in 'ABC']) <= BUDGET)

# Objectif principal : maximiser impact
impact_obj = opt_lex.maximize(impact_total)
# Objectif secondaire : minimiser cout (ne s'active que parmi les solutions d'impact max)
cout_obj = opt_lex.minimize(cout_total)

if opt_lex.check() == sat:
    m = opt_lex.model()
    selection = sorted([p for p in 'ABC' if m[x[p]].as_long() == 1])
    impact_val = m.evaluate(impact_total).as_long()
    cout_val = m.evaluate(cout_total).as_long()
    print(f'Lex : selection = {selection} (impact = {impact_val}, cout = {cout_val} k\u20ac)')
    print(f'  -> B + C (impact 18, cout 12 k\u20ac) : la seule combinaison a impact 18.')
    print(f'  -> Cout secondaire = 12 k\u20ac (le seul faisable pour impact = 18).')
else:
    print('Lex : aucune solution')

Lex : selection = ['B', 'C'] (impact = 18, cout = 12 k€)
  -> B + C (impact 18, cout 12 k€) : la seule combinaison a impact 18.
  -> Cout secondaire = 12 k€ (le seul faisable pour impact = 18).


### Interprétation : optimisation lexicographique

La stratégie lex donne une **seule solution** : `B + C` (coût 12 k€, impact 18). Le solveur commence par chercher l'impact maximum faisable (18), et parmi toutes les combinaisons qui atteignent cet impact, il choisit celle de coût minimum. C'est la même solution qu'un décideur rationnel prendrait s'il considère l'impact comme absolument prioritaire.

**Forces** :

- Décision **non-ambiguë** : une seule solution sort, pas d'arbitrage subjectif du décideur.
- Ordre d'importance **explicite** : si A est plus important que B, le solveur respecte cette hiérarchie strictement.
- Le solveur Z3 `Optimize()` implémente cette stratégie nativement via `maximize(...)` puis `minimize(...)`.

**Faiblesses** :

- **Cécité aux autres compromis** : si l'impact maximum est 18 mais qu'une solution à impact 17 coûte 5 k€ de moins,   lex l'ignore. Le décideur ne voit pas le tradeoff.
- **Sensible à l'ordre des critères** : inverser l'ordre (coût d'abord, puis impact) donnerait une solution   différente. C'est cohérent (les critères ne sont pas symétriques), mais cela oblige à un choix d'arbitrage *a priori*.

**Quand l'utiliser** : quand les critères ont une **hiérarchie claire** (e.g. sécurité d'abord, performance ensuite) ou quand le décideur veut une réponse claire et non-discutable.

In [3]:
# Mesure 2 -- enumeration du front de Pareto : on garde les solutions non-dominees
# Strategie : iterer, a chaque etape ajouter une contrainte d'exclusion, jusqu'a plus de solution
solutions = []
opt_p = Optimize()
for p in 'ABC':
    opt_p.add(0 <= x[p], x[p] <= 1)
opt_p.add(Sum([x[p] * couts[p] for p in 'ABC']) <= BUDGET)

# On maximise l'impact pour trouver le point extreme 'impact max'
h = opt_p.maximize(impact_total)

while opt_p.check() == sat:
    m = opt_p.model()
    sel = tuple(sorted([p for p in 'ABC' if m[x[p]].as_long() == 1]))
    imp = m.evaluate(impact_total).as_long()
    cout = m.evaluate(cout_total).as_long()
    solutions.append((sel, imp, cout))
    # Contrainte d'exclusion : empecher de retourner exactement a cette solution
    opt_p.add(Or([x[p] != m[x[p]].as_long() for p in 'ABC']))

print(f'Pareto : {len(solutions)} solutions non-dominnees trouvees\n')
print(f'{"Selection":<12} {"Impact":>8} {"Cout (k\u20ac)":>12}')
print('-' * 36)
for sel, imp, cout in solutions:
    label = 'aucun' if not sel else '+'.join(sel)
    print(f'{label:<12} {imp:>8} {cout:>12}')

# Filtrer les Pareto-optimales : eliminer les solutions qui en dominent une autre
from itertools import combinations
def domine(s1, s2):
    """s1 domine s2 si s1 a cout <= et impact >=, avec au moins un strictement meilleur."""
    return s1[2] <= s2[2] and s1[1] >= s2[1] and (s1[2] < s2[2] or s1[1] > s2[1])

pareto = []
for s in solutions:
    # Exclure la solution vide (ne rien selectionner) : trivialement Pareto-non-dominee
    # (cout 0 minimise le cout, mais impact 0 n'ameliore rien) -- on la filtre pour
    # ne garder que les selections reelles.
    if len(s[0]) == 0:
        continue
    if not any(domine(t, s) for t in solutions if t != s and len(t[0]) > 0):
        pareto.append(s)
print(f'\nApres filtrage (selection non vide + non-dominance) : {len(pareto)} solutions Pareto-optimales')

Pareto : 7 solutions non-dominnees trouvees

Selection      Impact    Cout (k€)
------------------------------------
B+C                18           12
A+C                16           11
A+B                14            9
C                  10            7
B                   8            5
A                   6            4
aucun               0            0

Apres filtrage (selection non vide + non-dominance) : 6 solutions Pareto-optimales


### Interprétation : front de Pareto

Les **6 solutions** trouvées (B+C, A+C, A+B, C, B, A) sont **toutes Pareto-optimales** : aucune n'est strictement dominée par une autre. Cela vient du fait que les 3 projets ont des **ratios impact/coût différents** (A=1.5, B=1.6, C=1.43) et que chaque solution occupe une niche sur le front.

**Cas dominé vs Pareto-optimal** : on parle de *domination* quand une solution `s1` est meilleure que `s2` sur **tous** les critères simultanément. Ici, par exemple, *aucune* solution n'est meilleure que toutes les autres sur (coût, impact) à la fois -- le problème est multi-objectif authentique, et le front est large.

**Forces** :

- **Visibilité complète des compromis** : le décideur voit *toutes* les options rationnelles (coût min, impact max, intermédiaires).
- **Pas d'arbitrage caché** : le poids de chaque critère n'est pas figé -- le décideur arbitre *après* avoir vu le front.
- **Fondement théorique solide** : la dominance Pareto est l'outil standard en recherche opérationnelle et en économie du bien-être.

**Faiblesses** :

- Le front peut être **trop large** (ici 6 solutions, mais un problème industriel peut en avoir des milliers).
- **Pas de hiérarchie** : le front est un *ensemble*, pas une recommandation -- le décideur doit choisir *dans* le front.
- Coûteux à énumérer : il faut résoudre le problème plusieurs fois (une fois par solution du front).

**Quand l'utiliser** : en phase exploratoire (cadrage d'un nouveau problème), ou quand le décideur veut **voir** le paysage des compromis avant de pondérer.

In [4]:
# Mesure 3 -- somme ponderee : on agrege impact - alpha * cout, et on fait varier alpha
alphas = [0.0, 0.5, 1.0, 1.3, 1.5, 2.0, 5.0, 10.0]
resultats = []

for alpha in alphas:
    opt_w = Optimize()
    for p in 'ABC':
        opt_w.add(0 <= x[p], x[p] <= 1)
    opt_w.add(Sum([x[p] * couts[p] for p in 'ABC']) <= BUDGET)
    # Objectif : maximiser impact - alpha * cout (equivalent a maximiser impact seul si alpha=0,
    # ou minimiser cout seul si alpha est tres grand)
    obj = impact_total - alpha * cout_total
    h_w = opt_w.maximize(obj)
    if opt_w.check() == sat:
        m = opt_w.model()
        sel = sorted([p for p in 'ABC' if m[x[p]].as_long() == 1])
        imp = m.evaluate(impact_total).as_long()
        cout = m.evaluate(cout_total).as_long()
        score = imp - alpha * cout
        label = 'aucun' if not sel else '+'.join(sel)
        resultats.append((alpha, label, imp, cout, score))

print(f'{"alpha":>6} {"Selection":<12} {"Impact":>8} {"Cout":>6} {"Score":>8}  Lecture')
print('-' * 70)
for alpha, label, imp, cout, score in resultats:
    if label == 'aucun':
        lecture = 'solution vide (ne rien faire)'
    elif alpha == 0.0:
        lecture = 'max impact seul'
    else:
        lecture = 'compromis impact / cout'
    print(f'{alpha:>6.2f} {label:<12} {imp:>8} {cout:>5}k {score:>8.2f}  {lecture}')

# --- Lecture derivee -----------------------------------------------------------
# Les chiffres de la lecture sont CALCULES, pas recopies a la main : on enumere
# toutes les selections faisables, puis, pour chaque alpha, l'ensemble des
# selections optimales. C'est ce qui evite qu'une prose contredise le tableau
# ci-dessus -- et c'est aussi ce qui revele que le front de Pareto contient des
# points que la somme ponderee n'atteint jamais.
from itertools import combinations

FASIBLES = []
for k in range(len('ABC') + 1):
    for combo in combinations('ABC', k):
        cout_combo = sum(couts[p] for p in combo)
        if cout_combo <= BUDGET:
            FASIBLES.append((combo, sum(impacts[p] for p in combo), cout_combo))

def score_de(sel, alpha):
    """Score de la selection sel (impact, cout) pour un poids alpha du cout."""
    return sel[1] - alpha * sel[2]

def optimaux(alpha):
    """Ensemble des selections optimales pour un alpha donne."""
    meilleur = max(score_de(s, alpha) for s in FASIBLES)
    return [s for s in FASIBLES if abs(score_de(s, alpha) - meilleur) < 1e-9]

def nom(sel):
    return 'vide' if not sel[0] else '+'.join(sel[0])

print('\nLecture : la solution glisse le long du front de Pareto quand alpha monte.')
for alpha in alphas:
    details = ' | '.join(f'{nom(s)} (impact {s[1]}, cout {s[2]} kEUR, score {score_de(s, alpha):.2f})'
                         for s in optimaux(alpha))
    print(f'  alpha = {alpha:<5} : {details}')

print("\nSeuils d'alpha deduits des egalites de score :")
print("  (une selection est optimale exactement sur l'intervalle indique)")
for s in FASIBLES:
    borne_inf, borne_sup = 0.0, float('inf')
    for t in FASIBLES:
        if t is s:
            continue
        ecart_cout = s[2] - t[2]
        if ecart_cout == 0:
            continue
        alpha_egalite = (s[1] - t[1]) / ecart_cout
        if ecart_cout > 0:
            borne_sup = min(borne_sup, alpha_egalite)
        else:
            borne_inf = max(borne_inf, alpha_egalite)
    if borne_inf <= borne_sup:
        print(f'  {nom(s):<8} : alpha dans [{borne_inf:.4g}, {borne_sup:.4g}]')


 alpha Selection      Impact   Cout    Score  Lecture
----------------------------------------------------------------------
  0.00 B+C                18    12k    18.00  max impact seul
  0.50 B+C                18    12k    12.00  compromis impact / cout
  1.00 B+C                18    12k     6.00  compromis impact / cout
  1.30 B+C                18    12k     2.40  compromis impact / cout
  1.50 B                   8     5k     0.50  compromis impact / cout
  2.00 aucun               0     0k     0.00  solution vide (ne rien faire)
  5.00 aucun               0     0k     0.00  solution vide (ne rien faire)
 10.00 aucun               0     0k     0.00  solution vide (ne rien faire)

Lecture : la solution glisse le long du front de Pareto quand alpha monte.
  alpha = 0.0   : B+C (impact 18, cout 12 kEUR, score 18.00)
  alpha = 0.5   : B+C (impact 18, cout 12 kEUR, score 12.00)
  alpha = 1.0   : B+C (impact 18, cout 12 kEUR, score 6.00)
  alpha = 1.3   : B+C (impact 18, cout 12 kEUR,

### Interprétation : somme pondérée

La somme pondérée `maximiser(impact - α * coût)` fait varier la solution selon `α` :

- `α = 0` : on ignore le coût, on maximise l'impact → **B+C** (impact 18).
- `α = 0.5` à `α = 1.0` : compromis équilibré → **B+C** (le score 18 - α × 12 reste le plus élevé).
- `α = 1.3` : le coût commence à peser, mais pas encore assez pour changer de solution → **B+C** (score 2.4),   devant **A+B** (score 2.3). Les deux s'égalisent exactement à **α = 4/3 ≈ 1.33** : c'est le seuil où B+C   cède la place à A+B.
- `α = 1.5` : **A+B** et **B** sont les deux solutions optimales, à score 0.5 chacune (le solveur n'en retourne   qu'une). A, B+C et la solution vide sont à 0 ; A+C et C sont à -0.5. Le coût pèse alors assez pour que le   projet le plus rentable (B) concurrence le couple A+B.
- `α > 8/5 = 1.6` : la **solution vide** devient optimale (coût 0, impact 0, score 0). À `α = 1.6` elle est à   égalité avec B, et pour `α ≥ 2` tous les scores non vides sont strictement négatifs. Le modèle dit   littéralement *"ne rien sélectionner"*.
- `α = 10` : la solution vide reste optimale (coût >> impact, plus aucun projet ne vaut son coût).

**Forces** :

- **Simple** : un seul paramètre à régler (le vecteur de poids), une seule résolution par jeu de poids.
- **Continuité** : faire varier `α` continûment fait *glisser* la solution le long du front de Pareto.
- **Sensibilité** : on peut étudier comment la décision réagit à un changement d'arbitrage du décideur.

**Faiblesses** :

- **Pas concave → trous** : la somme pondérée ne parcourt que l'**enveloppe concave supérieure** du front de   Pareto. Ici, parmi les 6 solutions Pareto-optimales, seules **B+C**, **A+B** et **B** ont un intervalle d'α où   elles sont optimales ; **A, C et A+C** sont Pareto-optimales mais **jamais choisies** par la somme pondérée   (le « trou » typique des points non supportés). Un front concave serait, lui, entièrement exploré — ce n'est   pas garanti en général.
- **Solution vide possible** : si le poids du coût devient trop fort, la solution vide (ne rien faire) devient   optimale (ici au-delà de `α = 8/5`). C'est cohérent mathématiquement, mais c'est un signal d'alarme : le décideur   veut sans doute dépenser le budget, et la modélisation doit intégrer cette intention (par exemple via une   contrainte `somme(x_p) >= 1`).
- **Interprétabilité du poids** : dire "α = 2" n'a pas de signification concrète pour un décideur non technique.
- **Tirage au sort des poids** : pour échantillonner le front de Pareto, on tire `α` au hasard -- certains points   du front sont sur-représentés (concavité), d'autres sous-représentés.

**Quand l'utiliser** : quand le décideur peut exprimer un arbitrage numérique (même grossier) entre critères, ou pour faire une **analyse de sensibilité** (que se passe-t-il si je change d'avis ?). **Attention** : ajouter une contrainte `au moins 1 projet` si l'objectif du décideur est de dépenser le budget.


In [5]:
# Exercice 1 -- Optimisation lexicographique sur 4 projets
#
# Probleme : 4 projets D, E, F, G avec les caracteristiques suivantes :
#   D : cout 3, impact 5
#   E : cout 6, impact 9
#   F : cout 4, impact 7
#   G : cout 5, impact 8
# Budget total : 14 k€.
#
# Coder une optimisation lexicographique qui maximise l'impact d'abord, puis minimise le cout.
# Affichez la selection finale et l'impact/cout associes.

def exercice_lex_4vars():
    """A completer : coder la strategie lex pour 4 projets."""
    # TODO etudiant : creer l'instance Optimize(), poser les contraintes 0 <= x_p <= 1,
    # ajouter la contrainte budget, puis maximiser impact puis minimiser cout.
    # Afficher la selection finale.
    pass

exercice_lex_4vars()
print('Exercice 1 : voir cellule ci-dessus. Selection attendue = ? (a calculer)')

Exercice 1 : voir cellule ci-dessus. Selection attendue = ? (a calculer)


In [6]:
# Exercice 2 -- Enumeration du front de Pareto sur 4 projets
#
# Reprenez les memes 4 projets D, E, F, G que l'exercice 1.
# Enumerez TOUTES les solutions Pareto-optimales (selection, impact, cout).
# Affichez le nombre de solutions et le tableau resultat.

def exercice_pareto_4vars():
    """A completer : enumerer le front de Pareto pour 4 projets."""
    # TODO etudiant : reprendre la strategie d'enumeration de la mesure 2 (cellule 4),
    # l'adapter aux 4 projets D/E/F/G avec leur budget 14 k€.
    pass

exercice_pareto_4vars()
print('Exercice 2 : voir cellule ci-dessus. Nombre de solutions Pareto = ? (a calculer)')

Exercice 2 : voir cellule ci-dessus. Nombre de solutions Pareto = ? (a calculer)


In [7]:
# Exercice 3 -- Somme ponderee sur 4 projets avec 4 poids differents
#
# Reprenez les 4 projets D, E, F, G. Testez 4 valeurs de alpha (par exemple 0, 1, 2, 5).
# Pour chaque alpha, affichez la selection choisie par le solveur.
# Concluez : pour quelle valeur de alpha la selection change-t-elle ?

def exercice_weighted_4poids():
    """A completer : faire varier alpha et observer la selection optimale."""
    # TODO etudiant : boucle sur 4 valeurs de alpha, appel a Optimize() a chaque fois,
    # afficher (alpha, selection, impact, cout).
    pass

exercice_weighted_4poids()
print('Exercice 3 : voir cellule ci-dessus. Verdict attendu = "alpha=0 -> D+E+G (impact 22) ; '
      'alpha=1 -> D+E+G, ex aequo avec D+E+F et D+F+G (score 8) ; alpha=2 et alpha=5 -> solution vide '
      '(aucun projet, score 0)" ou similaire.')


Exercice 3 : voir cellule ci-dessus. Verdict attendu = "alpha=0 -> D+E+G (impact 22) ; alpha=1 -> D+E+G, ex aequo avec D+E+F et D+F+G (score 8) ; alpha=2 et alpha=5 -> solution vide (aucun projet, score 0)" ou similaire.


## Conclusion : arbre de décision pour choisir sa stratégie

Le tableau ci-dessous résume quand utiliser lex, Pareto ou weighted, en fonction du contexte du décideur.

| Situation du décideur | Stratégie recommandée | Justification |
|----------------------|----------------------|---------------|
| **Hiérarchie claire des critères** (e.g. sécurité > coût > performance) | **Lexicographique** | L'ordre d'importance est explicite ; une seule solution sort, pas d'arbitrage subjectif |
| **Cadrage exploratoire** (premier contact avec un nouveau problème) | **Pareto** | Le front complet révèle la structure des compromis ; le décideur voit *avant* de pondérer |
| **Analyse de sensibilité** (que se passe-t-il si je change d'avis ?) | **Somme pondérée avec sweep sur α** | On observe comment la solution glisse le long du front ; utile pour le décideur hésitant |
| **Décideur non technique, arbitrage grossier** | **Somme pondérée** | Facile à expliquer ("je préfère 2 fois plus le coût à l'impact") |
| **Front non-concave** (cas pathologique) | **Pareto** | La somme pondérée ne peut pas atteindre les points non-concaves ; seul Pareto les capture |

**Take-away** : les trois stratégies sont **complémentaires**, pas concurrentes. Dans la pratique, on commence par Pareto pour *voir* le paysage, on teste une ou deux valeurs de `α` pour sentir la *sensibilité*, et on finit par lex pour *trancher* quand le décideur a une hiérarchie claire en tête.

**Pour aller plus loin** :

- **Z3-06 Advanced Optimization** : explore d'autres facettes (soft constraints, weighted MaxSAT).
- **Z3-12 Real Arithmetic** : si les variables sont réelles (pas seulement entières 0/1), les solutions   deviennent continues et le front est un *polyèdre* dans l'espace des objectifs.
- **Documentation Z3 `Optimize()`** : https://microsoft.github.io/z3guide/programming/Z3%20JavaScript%20Examples/optimization.html